In [17]:
import pandas as pd

g3n = pd.read_csv('res/G3N_energy_consumtion_devset.csv')
g3n_full_wh = g3n.groupby(["iteration"])["wh_trap"].sum().reset_index()
g3n_global_wh = g3n.query('type == "global"').groupby(["iteration"])["wh_trap"].sum().reset_index()
g3n_cluster_wh = g3n.query('type == "cluster"').groupby(["iteration"])["wh_trap"].sum().reset_index()

q3 = pd.read_csv('res/Q3_energy_consumtion_devset.csv')
q3_full_wh = q3.groupby(["iteration"])["wh_trap"].sum().reset_index()
q3_global_wh = q3.query('type == "global"').groupby(["iteration"])["wh_trap"].sum().reset_index()
q3_cluster_wh = q3.query('type == "cluster"').groupby(["iteration"])["wh_trap"].sum().reset_index()

'G3N', g3n_full_wh, 'Q3', q3_full_wh

('G3N',
    iteration      wh_trap
 0          0  1562.620833
 1          1  1567.853333
 2          2  1566.773333
 3          3  1567.754167
 4          4  1565.301944
 5          5  1567.605833
 6          6  1564.526111
 7          7  1563.999167
 8          8  1563.705000
 9          9  1562.706111,
 'Q3',
    iteration     wh_trap
 0          0  611.145556
 1          1  612.600833
 2          2  612.811389
 3          3  612.794167
 4          4  612.527778
 5          5  612.603333
 6          6  612.866667
 7          7  612.305000
 8          8  612.641667
 9          9  611.273611)

In [22]:
def summarize(x, confidence=0.95):
    n = len(x)
    mean = np.mean(x)
    std = np.std(x, ddof=1)
    median = np.median(x)
    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    iqr = q3 - q1
    cv = std / mean * 100

    alpha = 1 - confidence
    tcrit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    sem = std / np.sqrt(n)
    ci_low = mean - tcrit * sem
    ci_high = mean + tcrit * sem

    return {
        "n": n,
        "mean": mean,
        "std": std,
        "median": median,
        "IQR": iqr,
        "CV_%": cv,
        "CI95_low": ci_low,
        "CI95_high": ci_high,
    }

summary_df = pd.DataFrame({
    "Gemma": summarize(g3n_full_wh['wh_trap']),
    "Qwen": summarize(q3_full_wh['wh_trap']),
}).T

summary_df.round(4)

,n,mean,std,median,IQR,CV_%,CI95_low,CI95_high
Gemma,10.0,1565.2846,2.0755,1564.9140,3.6192,0.1326,1563.7999,1566.7693
Qwen,10.0,612.3570,0.6264,612.6021,0.3953,0.1023,611.9089,612.8051


In [23]:
import numpy as np
from scipy import stats

# ============================================================
# BASIC DESCRIPTIVE STATISTICS
# ============================================================

def summarize(x, confidence=0.95):
    n = len(x)
    mean = np.mean(x)
    std = np.std(x, ddof=1)             # sample std
    median = np.median(x)
    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    iqr = q3 - q1
    cv = std / mean * 100               # coefficient of variation in %

    # t-based CI for mean
    alpha = 1 - confidence
    tcrit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    sem = std / np.sqrt(n)
    ci_low = mean - tcrit * sem
    ci_high = mean + tcrit * sem

    return {
        "n": n,
        "mean": mean,
        "std": std,
        "median": median,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "cv_percent": cv,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }

# ============================================================
# BOOTSTRAP CI FOR DIFFERENCE OF MEANS
# ============================================================

def bootstrap_mean_diff_ci(x, y, n_boot=10000, confidence=0.95, random_state=42):
    rng = np.random.default_rng(random_state)
    diffs = np.empty(n_boot)

    for i in range(n_boot):
        xb = rng.choice(x, size=len(x), replace=True)
        yb = rng.choice(y, size=len(y), replace=True)
        diffs[i] = np.mean(xb) - np.mean(yb)

    alpha = 1 - confidence
    low = np.percentile(diffs, 100 * alpha / 2)
    high = np.percentile(diffs, 100 * (1 - alpha / 2))
    return np.mean(x) - np.mean(y), low, high

# ============================================================
# OPTIONAL: NORMALITY CHECK
# Small n => interpret carefully
# ============================================================

def normality_check(x, name):
    stat, p = stats.shapiro(x)
    print(f"{name} Shapiro-Wilk p = {p:.4f}")
    if p < 0.05:
        print(f"  -> {name}: deviation from normality may be present.")
    else:
        print(f"  -> {name}: no strong evidence against normality.")

# ============================================================
# EFFECT SIZE
# ============================================================

def cohens_d_independent(x, y):
    nx = len(x)
    ny = len(y)
    sx = np.std(x, ddof=1)
    sy = np.std(y, ddof=1)

    pooled_std = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    d = (np.mean(x) - np.mean(y)) / pooled_std
    return d

# ============================================================
# RUN ANALYSIS
# ============================================================
gemma = g3n_full_wh['wh_trap']
qwen = q3_full_wh['wh_trap']

gemma_stats = summarize(gemma)
qwen_stats = summarize(qwen)

print("=== DESCRIPTIVE STATISTICS ===")
for name, s in [("Gemma", gemma_stats), ("Qwen", qwen_stats)]:
    print(f"\n{name}")
    print(f"  n             = {s['n']}")
    print(f"  mean          = {s['mean']:.4f}")
    print(f"  std           = {s['std']:.4f}")
    print(f"  median        = {s['median']:.4f}")
    print(f"  IQR           = {s['iqr']:.4f} (Q1={s['q1']:.4f}, Q3={s['q3']:.4f})")
    print(f"  CV            = {s['cv_percent']:.2f}%")
    print(f"  95% CI mean   = [{s['ci_low']:.4f}, {s['ci_high']:.4f}]")

print("\n=== NORMALITY CHECK ===")
normality_check(gemma, "Gemma")
normality_check(qwen, "Qwen")

print("\n=== BETWEEN-MODEL COMPARISON ===")

# Welch t-test (does not assume equal variances)
t_stat, t_p = stats.ttest_ind(gemma, qwen, equal_var=False)
print(f"Welch t-test: t = {t_stat:.4f}, p = {t_p:.6f}")

# Mann-Whitney U test (nonparametric)
u_stat, u_p = stats.mannwhitneyu(gemma, qwen, alternative="two-sided")
print(f"Mann-Whitney U: U = {u_stat:.4f}, p = {u_p:.6f}")

# Bootstrap CI for mean difference
diff, diff_low, diff_high = bootstrap_mean_diff_ci(gemma, qwen)
print(f"Mean difference (Gemma - Qwen) = {diff:.4f}")
print(f"Bootstrap 95% CI of difference = [{diff_low:.4f}, {diff_high:.4f}]")

# Relative difference
rel_diff = (np.mean(gemma) - np.mean(qwen)) / np.mean(gemma) * 100
print(f"Relative reduction of Qwen vs Gemma = {rel_diff:.2f}%")

# Effect size
d = cohens_d_independent(gemma, qwen)
print(f"Cohen's d = {d:.4f}")

=== DESCRIPTIVE STATISTICS ===

Gemma
  n             = 10
  mean          = 1565.2846
  std           = 2.0755
  median        = 1564.9140
  IQR           = 3.6192 (Q1=1563.7785, Q3=1567.3977)
  CV            = 0.13%
  95% CI mean   = [1563.7999, 1566.7693]

Qwen
  n             = 10
  mean          = 612.3570
  std           = 0.6264
  median        = 612.6021
  IQR           = 0.3953 (Q1=612.3607, Q3=612.7560)
  CV            = 0.10%
  95% CI mean   = [611.9089, 612.8051]

=== NORMALITY CHECK ===
Gemma Shapiro-Wilk p = 0.1649
  -> Gemma: no strong evidence against normality.
Qwen Shapiro-Wilk p = 0.0024
  -> Qwen: deviation from normality may be present.

=== BETWEEN-MODEL COMPARISON ===
Welch t-test: t = 1389.9770, p = 0.000000
Mann-Whitney U: U = 100.0000, p = 0.000183
Mean difference (Gemma - Qwen) = 952.9276
Bootstrap 95% CI of difference = [951.6628, 954.1882]
Relative reduction of Qwen vs Gemma = 60.88%
Cohen's d = 621.6166
